<a href="https://colab.research.google.com/github/ysau/PIML/blob/main/TimeSeries_Demo_LSTM_Transformer_Anomaly_UQ.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Time-Series Demo: LSTM vs Transformer + Anomaly Detection + Uncertainty (Colab-ready)

**What this notebook does**
1. Generate synthetic sensor time series with drift + injected anomalies.
2. Train **LSTM** and **Transformer** forecasters and compare.
3. Train a simple **Autoencoder** for unsupervised anomaly detection.
4. Add **Uncertainty** via Monte Carlo Dropout around the forecaster.
5. Plot forecasts, anomaly scores, and uncertainty bands.

**How to use** (Colab):
1. Runtime → Change runtime type → **GPU** (if available).
2. Run the setup cell to install packages (if needed).
3. Execute cells in sequence.

*This is a compact, teaching-focused demo—not SOTA. Swap in real data or HF time-series models later.*

In [ ]:
# --- Setup (Colab-friendly) ---
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip -q install torch torchvision torchaudio transformers==4.44.2 --upgrade
    !pip -q install scikit-learn matplotlib
print('Setup complete.')

In [ ]:
import math, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
device

## 1) Synthetic data with drift and anomalies
We create a base sine wave with slowly varying amplitude/phase and inject spikes + level-shifts as anomalies.

In [ ]:
def make_series(n=6000, drift_amp=0.0002, noise=0.05, spike_prob=0.002, step_prob=0.001):
    t = np.arange(n)
    # Slow drift in amplitude and phase
    amp = 1 + drift_amp * t
    phase = 0.001 * t
    base = amp * np.sin(0.05 * t + phase)
    x = base + noise * np.random.randn(n)
    labels = np.zeros(n, dtype=int)  # 1 if anomaly
    # Spikes
    for i in range(n):
        if np.random.rand() < spike_prob:
            x[i] += np.random.choice([4, -4])
            labels[i] = 1
    # Level shifts (lasting for a while)
    i = 0
    while i < n:
        if np.random.rand() < step_prob:
            step = np.random.choice([2.0, -2.0])
            dur = np.random.randint(50, 150)
            x[i:i+dur] += step
            labels[i:i+dur] = 1
            i += dur
        i += 1
    return x.astype(np.float32), labels

x, y_anom = make_series()
plt.figure(figsize=(10,3))
plt.plot(x)
plt.title('Synthetic time series (with drift + anomalies)')
plt.show()

### Windowed dataset
We train forecasters to predict the next point from the previous `context_len` points.

In [ ]:
class WindowDataset(Dataset):
    def __init__(self, series, context_len=64, pred_horizon=1, stride=1):
        self.series = series
        self.context_len = context_len
        self.pred_horizon = pred_horizon
        self.idxs = []
        for i in range(0, len(series) - context_len - pred_horizon, stride):
            self.idxs.append(i)
    def __len__(self):
        return len(self.idxs)
    def __getitem__(self, idx):
        i = self.idxs[idx]
        x = self.series[i:i+self.context_len]
        y = self.series[i+self.context_len:i+self.context_len+self.pred_horizon]
        return torch.tensor(x).unsqueeze(-1), torch.tensor(y)

split = int(0.8*len(x))
train_x, test_x = x[:split], x[split:]

context_len = 64
pred_horizon = 1
train_ds = WindowDataset(train_x, context_len, pred_horizon)
test_ds  = WindowDataset(test_x, context_len, pred_horizon)
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)
test_dl  = DataLoader(test_ds, batch_size=256, shuffle=False)
len(train_ds), len(test_ds)

## 2) LSTM forecaster

In [ ]:
class LSTMForecaster(nn.Module):
    def __init__(self, input_size=1, hidden=64, layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden, num_layers=layers, batch_first=True)
        self.head = nn.Linear(hidden, 1)
        self.dropout = nn.Dropout(p=0.2)  # used for MC Dropout
    def forward(self, x):
        # x: (B, T, 1)
        out, _ = self.lstm(x)
        out = self.dropout(out[:, -1, :])
        return self.head(out).squeeze(-1)

def train_epoch(model, dl, opt):
    model.train()
    total = 0.0
    for xb, yb in dl:
        xb = xb.to(device)
        yb = yb.squeeze(-1).to(device)
        opt.zero_grad()
        pred = model(xb)
        loss = F.mse_loss(pred, yb)
        loss.backward()
        opt.step()
        total += loss.item() * xb.size(0)
    return total/len(dl.dataset)

def eval_epoch(model, dl):
    model.eval()
    total = 0.0
    with torch.no_grad():
        for xb, yb in dl:
            xb = xb.to(device)
            yb = yb.squeeze(-1).to(device)
            pred = model(xb)
            loss = F.mse_loss(pred, yb)
            total += loss.item() * xb.size(0)
    return total/len(dl.dataset)

lstm = LSTMForecaster().to(device)
opt = torch.optim.Adam(lstm.parameters(), lr=1e-3)
for epoch in range(5):
    tr = train_epoch(lstm, train_dl, opt)
    te = eval_epoch(lstm, test_dl)
    print(f"[LSTM] epoch {epoch+1}: train {tr:.4f} | test {te:.4f}")

## 3) Transformer forecaster (PyTorch `nn.TransformerEncoder`)

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=1000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, d_model)
    def forward(self, x):
        # x: (B, T, d_model)
        T = x.size(1)
        return x + self.pe[:, :T, :]

class TransformerForecaster(nn.Module):
    def __init__(self, d_model=64, nhead=4, num_layers=2, dim_feedforward=128):
        super().__init__()
        self.inp = nn.Linear(1, d_model)
        self.pos = PositionalEncoding(d_model)
        enc_layer = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward, batch_first=True)
        self.enc = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.dropout = nn.Dropout(p=0.2)  # for MC Dropout
        self.head = nn.Linear(d_model, 1)
    def forward(self, x):
        # x: (B, T, 1)
        h = self.inp(x)
        h = self.pos(h)
        h = self.enc(h)
        h = self.dropout(h[:, -1, :])
        return self.head(h).squeeze(-1)

trans = TransformerForecaster().to(device)
opt_t = torch.optim.Adam(trans.parameters(), lr=1e-3)
for epoch in range(5):
    tr = train_epoch(trans, train_dl, opt_t)
    te = eval_epoch(trans, test_dl)
    print(f"[Transformer] epoch {epoch+1}: train {tr:.4f} | test {te:.4f}")

## 4) Forecast visualization on a rolling window

In [ ]:
def forecast_one_step(model, series, start, context_len=64):
    model.eval()
    with torch.no_grad():
        ctx = torch.tensor(series[start:start+context_len]).float().unsqueeze(0).unsqueeze(-1).to(device)
        pred = model(ctx).item()
    return pred

def roll_forecast(model, series, context_len=64, steps=300):
    preds = []
    idx0 = len(series) - steps - context_len
    for i in range(steps):
        preds.append(forecast_one_step(model, series, idx0 + i, context_len))
    truth = series[-steps:]
    return np.array(preds), truth

steps = 400
pred_lstm, truth = roll_forecast(lstm, test_x, context_len, steps)
pred_trans, _ = roll_forecast(trans, test_x, context_len, steps)

plt.figure(figsize=(10,3))
plt.plot(truth, label='truth')
plt.plot(pred_lstm, label='lstm')
plt.plot(pred_trans, label='transformer')
plt.legend()
plt.title('One-step rolling forecast (test split)')
plt.show()

## 5) Autoencoder for anomaly detection (reconstruction error)

In [ ]:
class ConvAE(nn.Module):
    def __init__(self, channels=1):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv1d(channels, 8, 5, padding=2), nn.ReLU(),
            nn.Conv1d(8, 16, 5, padding=2), nn.ReLU(),
            nn.Conv1d(16, 32, 5, padding=2), nn.ReLU(),
        )
        self.dec = nn.Sequential(
            nn.Conv1d(32, 16, 5, padding=2), nn.ReLU(),
            nn.Conv1d(16, 8, 5, padding=2), nn.ReLU(),
            nn.Conv1d(8, channels, 5, padding=2),
        )
    def forward(self, x):  # x: (B, T, 1)
        x = x.transpose(1,2)
        z = self.enc(x)
        rec = self.dec(z)
        return rec.transpose(1,2)

ae = ConvAE().to(device)
opt_ae = torch.optim.Adam(ae.parameters(), lr=1e-3)
def train_ae_epoch(model, dl, opt):
    model.train()
    total=0
    for xb, _ in dl:
        xb = xb.to(device)
        opt.zero_grad()
        rec = model(xb)
        loss = F.mse_loss(rec, xb)
        loss.backward()
        opt.step()
        total += loss.item() * xb.size(0)
    return total/len(dl.dataset)

for epoch in range(3):
    tr = train_ae_epoch(ae, train_dl, opt_ae)
    print(f"[AE] epoch {epoch+1}: train recon MSE {tr:.4f}")

# Compute anomaly score on test set windows as recon error
ae.eval()
scores = []
with torch.no_grad():
    for xb, _ in test_dl:
        xb = xb.to(device)
        rec = ae(xb)
        err = F.mse_loss(rec, xb, reduction='none').mean(dim=(1,2)).cpu().numpy()
        scores.extend(err)
scores = np.array(scores)
plt.figure(figsize=(10,3))
plt.plot(scores)
plt.title('Anomaly score (reconstruction error) over test windows')
plt.show()

## 6) Uncertainty with Monte Carlo Dropout
We reuse the trained models and keep dropout **on** at inference (multiple passes) to estimate predictive variance.

In [ ]:
def mc_predict(model, series, context_len=64, steps=200, passes=20):
    model.train()  # enable dropout at inference!
    preds = []
    variances = []
    idx0 = len(series) - steps - context_len
    for i in range(steps):
        ctx = torch.tensor(series[idx0+i:idx0+i+context_len]).float().unsqueeze(0).unsqueeze(-1).to(device)
        outs = []
        for _ in range(passes):
            outs.append(model(ctx).item())
        m = np.mean(outs)
        v = np.var(outs)
        preds.append(m)
        variances.append(v)
    return np.array(preds), np.array(variances)

mc_steps = 250
pred_mean, pred_var = mc_predict(trans, test_x, context_len, steps=mc_steps, passes=30)
truth_mc = test_x[-mc_steps:]
std = np.sqrt(pred_var)
low = pred_mean - 2*std
high = pred_mean + 2*std

plt.figure(figsize=(10,3))
plt.plot(truth_mc, label='truth')
plt.plot(pred_mean, label='pred mean')
plt.fill_between(np.arange(len(pred_mean)), low, high, alpha=0.3, label='~95% band')
plt.legend()
plt.title('Transformer MC Dropout Uncertainty')
plt.show()

## 7) Notes & Next steps
- Swap synthetic data for real sensor data (ECG, RF pulses, etc.).
- Consider HF time-series models (Chronos/TimesFM) or PatchTST for stronger baselines.
- Replace ConvAE with LSTM/Transformer AE for sequence-aware reconstruction.
- Try conformal prediction for calibrated intervals; add an Isolation Forest baseline.